# IT3212 - Assignment 3 - Task 2: Preprocessing

Cleans the smoking/drinking health-screening dataset and produces a fixed
train/val/test split for the downstream tasks (T3-T10).

**Target:** predict smoking status (never vs. ever smoked) from objective
health measurements.

**Outputs** (written to `../data/assignment3/`):
- `task2_train.csv`, `task2_val.csv`, `task2_test.csv` (unscaled)
- `task2_metadata.json` (schema/contract for downstream tasks, including the
  scaling recipe)

The data is exported **unscaled** on purpose - see section 8.

## Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Load data, drop drinking, collapse target

Two decisions are applied at load time:

1. **Drop `DRK_YN` (drinking).** Drinking is a *self-reported behaviour*, not a
   health *measurement*. Keeping it would mean predicting one self-report from
   another, which is circular. The problem statement only refers to
   "demographic and health-related measurements", so drinking is excluded.

2. **Collapse the target to 2 classes.** `SMK_stat_type_cd` has three codes
   (1 = never, 2 = former, 3 = current). We frame the problem as
   **never vs. ever smoked**, so 2 and 3 are merged. The column is renamed to
   `smoking` because the values no longer correspond to the original codes.

The collapse happens here (rather than downstream) because the stratified
split below must use the *final* labels.

In [2]:
RAW_PATH = Path("../data/smoking_driking_dataset_Ver01.csv")
OUT_DIR = Path("../data/assignment3")
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_RAW = "SMK_stat_type_cd"
TARGET = "smoking"
RANDOM_STATE = 42

df = pd.read_csv(RAW_PATH)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

Loaded: 991,346 rows x 24 columns


In [3]:
# Drop drinking (self-reported behaviour, not a health measurement)
df = df.drop(columns=["DRK_YN"])

# Collapse target: 1 (never) -> 0, 2 (former) / 3 (current) -> 1 (ever)
df[TARGET] = df[TARGET_RAW].map({1.0: 0, 2.0: 1, 3.0: 1}).astype(int)
df = df.drop(columns=[TARGET_RAW])

print("Class balance (0 = never, 1 = ever):")
print(df[TARGET].value_counts().sort_index())
print(f"\nRatio: {df[TARGET].mean():.3f}")

Class balance (0 = never, 1 = ever):
smoking
0    602441
1    388905
Name: count, dtype: int64

Ratio: 0.392


## 2. Remove duplicate rows

26 duplicate rows were found. Without a unique identifier we cannot verify
whether these are distinct people with identical profiles or repeated entries.
Since they are a negligible share of the data (< 0.01%), we remove them to be
safe rather than risk duplicated observations skewing the models.

In [4]:
n_dup = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {n_dup} duplicate rows -> {df.shape[0]:,} rows remaining")

Removed 26 duplicate rows -> 991,320 rows remaining


## 3. Check missing values

The dataset contains no missing values. We still run the check explicitly so
the decision to take no action is documented rather than assumed.

In [5]:
n_missing = int(df.isnull().sum().sum())
print(f"Total missing values: {n_missing}")
print("No action needed." if n_missing == 0 else "Missing values present - handle before proceeding.")

Total missing values: 0
No action needed.


## 4. Remove sentinel / erroneous values

Some columns contain values that are not physically possible (e.g.
`waistline > 200`, `sight > 3`, `SGOT_AST > 2000`). These are data-entry
errors, not genuine outliers. Unlike IQR capping, these thresholds come from
physiological domain knowledge, so they are **not fitted to the data** and can
safely be applied to all rows before the split.

Thresholds are reused from Assignment 1.

In [6]:
sentinel_mask = (
    (df["waistline"] > 200) |              # 999 impossible for a waist measurement
    (df["sight_left"] > 3) |
    (df["sight_right"] > 3) |              # 9.9 outside plausible acuity scale
    (df["HDL_chole"] > 500) |              # 8110 biologically impossible
    (df["LDL_chole"] > 500) |              # 5119 biologically impossible
    (df["serum_creatinine"] > 15) |        # 98 biologically impossible
    (df["SGOT_AST"] > 2000) |              # 9999 sentinel value
    (df["SGOT_ALT"] > 2000) |              # 7210 implausible
    (df["gamma_GTP"] > 700)                # 999 likely sentinel
)

n_sentinel = int(sentinel_mask.sum())
df = df[~sentinel_mask].reset_index(drop=True)
print(f"Removed {n_sentinel:,} rows as erroneous -> {df.shape[0]:,} rows remaining")

Removed 6,788 rows as erroneous -> 984,532 rows remaining


## 5. Split into train / validation / test (80/10/10)

The split happens **before** outlier capping and scaling so that any bounds or
parameters derived from the data are learned from the training set only.

Splitting is stratified on the target so all three sets keep the same
never/ever proportion.

In [7]:
train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE, stratify=df[TARGET]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_STATE, stratify=temp_df[TARGET]
)

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s}: {len(part):>7,} rows  |  ever ratio = {part[TARGET].mean():.3f}")

train: 787,625 rows  |  ever ratio = 0.392
val  :  98,453 rows  |  ever ratio = 0.392
test :  98,454 rows  |  ever ratio = 0.392


## 6. Cap genuine outliers (IQR bounds from train only)

After removing the impossible values, the remaining extreme values are real
measurements. We cap (Winsorize) them at the IQR bounds instead of removing
them, so real patient data is kept.

**The bounds are computed on the training set only** and then applied to all
three sets. Computing them on the full dataset would let information from the
validation/test rows leak into preprocessing.

In [8]:
# Continuous columns to cap (exclude binary / coded categorical columns)
CATEGORICAL_COLS = ["sex", "hear_left", "hear_right", "urine_protein"]
cap_cols = [
    c for c in df.columns
    if c != TARGET and c not in CATEGORICAL_COLS
]

# Derive IQR bounds from the TRAINING set only
cap_bounds = {}
for col in cap_cols:
    q1 = train_df[col].quantile(0.25)
    q3 = train_df[col].quantile(0.75)
    iqr = q3 - q1
    cap_bounds[col] = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)

# Apply the train-derived bounds to all three sets
for part in (train_df, val_df, test_df):
    for col in cap_cols:
        lower, upper = cap_bounds[col]
        part[col] = part[col].clip(lower=lower, upper=upper)

print(f"Capped {len(cap_cols)} continuous columns using train-derived IQR bounds.")
print("Example bounds:")
for col in ["triglyceride", "gamma_GTP", "hemoglobin"]:
    lo, hi = cap_bounds[col]
    print(f"  {col:16s}: [{lo:.2f}, {hi:.2f}]")

Capped 18 continuous columns using train-derived IQR bounds.
Example bounds:
  triglyceride    : [-56.00, 288.00]
  gamma_GTP       : [-18.50, 73.50]
  hemoglobin      : [9.90, 18.70]


## 7. Encode categorical columns

- `sex` (Male/Female) and `hear_left` / `hear_right` (1/2) are binary -> map to 0/1.
- `urine_protein` (grades 1-6) is kept as an **ordinal number**, since the
  grades have a meaningful order (increasing protein level) and ~94% of values
  are grade 1, so one-hot encoding would only add near-empty columns.

Encoding is stateless (same output per value), so applying it after the split
is safe.

In [9]:
for part in (train_df, val_df, test_df):
    part["sex"] = part["sex"].map({"Male": 1, "Female": 0}).astype(int)
    part["hear_left"] = part["hear_left"].map({1.0: 0, 2.0: 1}).astype(int)
    part["hear_right"] = part["hear_right"].map({1.0: 0, 2.0: 1}).astype(int)

print("Encoding applied.")
train_df[["sex", "hear_left", "hear_right", "urine_protein", TARGET]].head()

Encoding applied.


,sex,hear_left,hear_right,urine_protein,smoking
490013,1,0,0,1.0,0
888646,1,0,0,1.0,1
640611,0,0,0,1.0,0
971062,0,0,0,1.0,0
785060,0,0,0,1.0,0


## 8. Scaling - recipe only, no scaler is fitted here

The exported data is left **unscaled** so downstream tasks can choose per
model: tree-based models (decision trees, random forest, boosting) do not need
scaling, while KNN, SVM and neural networks do.

**No scaler is fitted or saved.** Instead, the recipe is recorded below and in
`task2_metadata.json`. Downstream tasks fit a `StandardScaler` themselves, on the
**training set only**, using exactly these columns.

Why a recipe rather than a saved file:

- Fitting a `StandardScaler` on the fixed `task2_train.csv` is deterministic,
  so a saved file adds no information.
- It keeps the pipeline free of binary artifacts that must be kept in sync as
  later tasks change the feature set.
- The one rule that matters - **fit on train, never on train+val+test** - is
  stated explicitly here and in `task2_metadata.json`.

Binary and ordinal columns (`sex`, `hear_left`, `hear_right`, `urine_protein`)
are excluded: scaling a 0/1 flag or an ordinal grade is meaningless.

In [10]:
# Columns that a downstream scaler should cover: all continuous columns.
# The binary and ordinal columns are excluded by construction (see section 7).
BINARY_OR_ORDINAL = ["sex", "hear_left", "hear_right", "urine_protein"]

scale_cols = [
    c for c in train_df.columns
    if c != TARGET and c not in BINARY_OR_ORDINAL
]

print(f"{len(scale_cols)} continuous columns to scale (recipe recorded in task2_metadata.json):")
print("  " + ", ".join(scale_cols))

18 continuous columns to scale (recipe recorded in task2_metadata.json):
  age, height, weight, waistline, sight_left, sight_right, SBP, DBP, BLDS, tot_chole, HDL_chole, LDL_chole, triglyceride, hemoglobin, serum_creatinine, SGOT_AST, SGOT_ALT, gamma_GTP


## 9. Export the splits and metadata

Writes the three unscaled CSVs plus `task2_metadata.json`, which records the target
name, class mapping, column list and the scaling recipe.

`task2_metadata.json` is the schema contract for downstream tasks: it prevents
mistakes such as scaling the target or misreading the class labels.

In [11]:
train_df.to_csv(OUT_DIR / "task2_train.csv", index=False)
val_df.to_csv(OUT_DIR / "task2_val.csv", index=False)
test_df.to_csv(OUT_DIR / "task2_test.csv", index=False)

metadata = {
    "task": "task2_preprocessing",
    "target": TARGET,
    "target_classes": {"0": "never smoked", "1": "ever smoked (current or former)"},
    "dropped_columns": ["DRK_YN", TARGET_RAW],
    "feature_columns": [c for c in train_df.columns if c != TARGET],
    "scaling": {
        "applied_to_exports": False,
        "method": "StandardScaler",
        "columns": scale_cols,
        "excluded_columns": BINARY_OR_ORDINAL,
        "fit_on": "train only - never fit on train+val+test",
        "note": (
            "No scaler file is shipped. Downstream tasks fit a StandardScaler "
            "on task2_train.csv using exactly these columns."
        ),
    },
    "split": {"train": 0.80, "val": 0.10, "test": 0.10, "stratified_on": TARGET},
    "random_state": RANDOM_STATE,
    "row_counts": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
}

with open(OUT_DIR / "task2_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Exported:")
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name}")

Exported:
  metadata.json
  task2_metadata.json
  task2_test.csv
  task2_train.csv
  task2_val.csv
  task3_metadata.json
  task3_test.csv
  task3_train.csv
  task3_val.csv
  task4_metadata.json
  task4_test.csv
  task4_train.csv
  task4_val.csv


## 10. Sanity check

Reload the exports to confirm they are readable, unscaled and carry the
expected target distribution.

In [12]:
for name in ["train", "val", "test"]:
    d = pd.read_csv(OUT_DIR / f"task2_{name}.csv")
    print(f"task2_{name}.csv: {d.shape}  target mean = {d[TARGET].mean():.3f}")

print("\ntask2_metadata.json keys:", list(metadata.keys()))
pd.read_csv(OUT_DIR / "task2_train.csv").head()

task2_train.csv: (787625, 23)  target mean = 0.392
task2_val.csv: (98453, 23)  target mean = 0.392
task2_test.csv: (98454, 23)  target mean = 0.392

task2_metadata.json keys: ['task', 'target', 'target_classes', 'dropped_columns', 'feature_columns', 'scaling', 'split', 'random_state', 'row_counts']


,sex,age,height,weight,waistline,sight_left,sight_right,hear_left,hear_right,SBP,...,HDL_chole,LDL_chole,triglyceride,hemoglobin,urine_protein,serum_creatinine,SGOT_AST,SGOT_ALT,gamma_GTP,smoking
0,1,70,165.0,65.0,89.0,1.0,0.5,0,0,138.0,...,64.0,92.0,135.0,15.4,1.0,0.8,27.0,24.0,25.0,0
1,1,40,170.0,60.0,77.9,1.0,1.0,0,0,100.0,...,42.0,91.0,173.0,14.4,1.0,1.0,41.5,50.0,73.5,1
2,0,60,155.0,50.0,80.0,0.6,0.8,0,0,100.0,...,65.0,98.0,219.0,14.4,1.0,0.5,24.0,26.0,44.0,0
3,0,75,140.0,45.0,82.0,0.5,0.6,0,0,139.0,...,36.0,82.0,136.0,11.8,1.0,0.7,35.0,17.0,42.0,0
4,0,40,160.0,55.0,82.8,1.2,1.2,0,0,112.0,...,55.0,113.0,129.0,13.5,1.0,0.7,21.0,14.0,8.0,0
